# Solar Panel Segmentation - Mask R-CNN (Detectron2) Training & Evaluation
This notebook configures Detectron2 with PyTorch and CUDA inside Google Colab, loads COCO annotations for 2021-2025, and trains Mask R-CNN ResNet-50 FPN.

**Hardware Recommendation:** Google Colab GPU (T4 / V100 / A100).

## 1. Setup Environment & Detectron2
Detectron2 installs seamlessly against Colab's pre-installed PyTorch & CUDA.

In [ ]:
# Verify GPU and PyTorch
!nvidia-smi
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Install Detectron2 from Facebook Research repo
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

## 2. Clone Dataset from GitHub
Clones `https://github.com/Hinarashid220/FYP.git` to load the multi-year COCO dataset (2021–2025).

In [ ]:
import os
from pathlib import Path

# Clone dataset from GitHub repository if not present
if not os.path.exists("FYP") and not os.path.exists("Dataset/COCO"):
    print("Cloning dataset from GitHub repository (Hinarashid220/FYP)...")
    !git clone https://github.com/Hinarashid220/FYP.git
else:
    print("Dataset repository already available.")

# Locate COCO directory (works in Google Colab or local environment)
if os.path.exists("FYP/Dataset/COCO"):
    COCO_ROOT = Path("FYP/Dataset/COCO")
elif os.path.exists("Dataset/COCO"):
    COCO_ROOT = Path("Dataset/COCO")
else:
    COCO_ROOT = Path("FYP/Dataset/COCO")

print(f"COCO Dataset Root: {COCO_ROOT.resolve()}")

## 3. Dataset Registration
Registers individual years and merges all years (2021-2025) into `solar_all_train`, `solar_all_valid`, `solar_all_test`.

In [ ]:
import json
from pathlib import Path
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

YEARS = ["2021", "2022", "2023", "2024", "2025"]

def register_datasets():
    for year in YEARS:
        for split in ["train", "valid", "test"]:
            name = f"solar_{year}_{split}"
            img_dir = str(COCO_ROOT / year / split / "images")
            ann_file = str(COCO_ROOT / year / split / "_annotations.coco.json")
            if os.path.exists(ann_file) and name not in DatasetCatalog.list():
                register_coco_instances(name, {}, ann_file, img_dir)

    for split in ["train", "valid", "test"]:
        _register_combined(split)

def _register_combined(split):
    combined_name = f"solar_all_{split}"
    if combined_name in DatasetCatalog.list():
        return

    def loader():
        all_records = []
        img_id_offset = 0
        for year in YEARS:
            ann_file = COCO_ROOT / year / split / "_annotations.coco.json"
            img_dir = COCO_ROOT / year / split / "images"
            if not ann_file.exists():
                continue
            data = json.loads(ann_file.read_text())
            old_to_new = {}
            for img in data.get("images", []):
                new_id = img["id"] + img_id_offset
                old_to_new[img["id"]] = new_id
                all_records.append({
                    "file_name": str(img_dir / Path(img["file_name"]).name),
                    "image_id": new_id,
                    "height": img["height"],
                    "width": img["width"],
                    "annotations": [],
                })
            rec_map = {r["image_id"]: r for r in all_records if r["image_id"] >= img_id_offset}
            for ann in data.get("annotations", []):
                nid = old_to_new.get(ann["image_id"])
                if nid and nid in rec_map:
                    rec_map[nid]["annotations"].append({
                        "bbox": ann["bbox"],
                        "bbox_mode": 1,
                        "segmentation": ann["segmentation"],
                        "category_id": 0,
                        "iscrowd": ann.get("iscrowd", 0),
                    })
            if old_to_new:
                img_id_offset = max(old_to_new.values()) + 1
        return all_records

    DatasetCatalog.register(combined_name, loader)
    MetadataCatalog.get(combined_name).set(thing_classes=["solar_panel"])

register_datasets()
print("Datasets registered:", [d for d in DatasetCatalog.list() if d.startswith('solar')])

## 4. Model Training Configuration & Execution

In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader

class SolarTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "eval")
        return COCOEvaluator(dataset_name, output_dir=output_folder)

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")

cfg.DATASETS.TRAIN = ("solar_all_train",)
cfg.DATASETS.TEST = ("solar_all_valid",)

cfg.SOLVER.IMS_PER_BATCH = 4        # Batch size for Colab GPU
cfg.SOLVER.BASE_LR = 0.0025
cfg.SOLVER.MAX_ITER = 5000
cfg.SOLVER.STEPS = (3000, 4000)
cfg.SOLVER.GAMMA = 0.1
cfg.SOLVER.CHECKPOINT_PERIOD = 1000
cfg.SOLVER.WARMUP_ITERS = 500

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg.INPUT.MIN_SIZE_TRAIN = (400, 432, 480)
cfg.INPUT.MAX_SIZE_TRAIN = 512
cfg.INPUT.MIN_SIZE_TEST = 432
cfg.INPUT.MAX_SIZE_TEST = 512
cfg.DATALOADER.NUM_WORKERS = 2

cfg.OUTPUT_DIR = "runs/solar-seg/mask-rcnn"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Start training
trainer = SolarTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## 5. Evaluation on Test Set

In [ ]:
cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
cfg.DATASETS.TEST = ("solar_all_test",)
predictor = DefaultPredictor(cfg)
evaluator = COCOEvaluator("solar_all_test", output_dir=cfg.OUTPUT_DIR)
test_loader = build_detection_test_loader(cfg, "solar_all_test")
results = inference_on_dataset(predictor.model, test_loader, evaluator)

print("=" * 60)
print("Mask R-CNN Test Set Results:")
if "segm" in results:
    print(f"  Segmentation mAP50:    {results['segm']['AP50']:.2f}")
    print(f"  Segmentation mAP50-95: {results['segm']['AP']:.2f}")
if "bbox" in results:
    print(f"  BBox mAP50:            {results['bbox']['AP50']:.2f}")
    print(f"  BBox mAP50-95:         {results['bbox']['AP']:.2f}")
print("=" * 60)

## 6. Export Results

In [ ]:
!zip -r mask-rcnn-results.zip runs/solar-seg/mask-rcnn